# Calculate percentage of columns covered by 70% in ProteinGym1
### A good MSA (similar to that from the EVmutation paper):
1. $>70\%$ of the columns covered by at least 70% of the sequences in the final alignment
2. Neff/L in 1-100, with some room for flexibility (<1 is ok after optimization)
3. If the two objectives above are conflicting, priority is given to 2
4. Additionally, for the purpose of PG2: each .m2a file should have
- more than 100 sequences
- less than 2M sequences (usually should be the case when Neff/L < 100)
- separately list the design sequences
5. Remove alignments that align to $<70\%$ of the length of the target sequence --> can change this to $<50\%$

## Check if we need to update selected bitscores

In [8]:
import sys
from glob import glob
import numpy as np
from os.path import basename, dirname, exists, isdir
import pandas as pd
import os
import shutil

There are 2 directories with MSA results:
1. From Tranception - 152 entries (93 are also in list 2) - these have bitscore 0.1, ..., 0.9 - `_alignment_statistics.csv` doesn't have Neff/L and min colcov varies - have evcouplings models
2. From Navami - 186 entries (all the uniprot id in PG final reference file) - these have bitscore 0.03, 0.04, 0.05, 0.1, 0.3, 0.5 - `_alignment_statistics.csv` all have Neff/L and min colcov=50 - spearman already calculated in a separate file (downloaded)

In [7]:
# tranception (may be outdated)
msa_dir1 = "/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym/EVCouplings/EVCouplings_runs/alignment_runs"
# Navami (seqcov 50 colcov 50)
msa_dir2 = "/n/groups/marks/projects/viral_families/notebooks/navami/followupNeffAnalysis/PG_analysis/evh/output/uniref100"

### Get the MSA stats from Navami's output

In [24]:
def aggregate_neff_l_colcov(msa_dir=msa_dir2):
    stats_df = {'DMS_ID': [], 'bitscore': [], 'num_seqs': [], 'Neff/L': [], 'Colcov50': []}
    for sample_path in glob(msa_dir + "/*"):
        sample = basename(sample_path)
        output_dirs = [p for p in glob(sample_path + '/' + sample + "_b*") if os.path.isdir(p)]
        for output_dir in output_dirs:
            output_name = basename(output_dir)  
            bitscore = output_name.removeprefix(f"{sample}_b")
            stat_file = output_dir + "/align/" + output_name + '_alignment_statistics.csv'
            try:
                stat_df = pd.read_csv(stat_file)
                neff, length, num_seqs, perc_cov = stat_df.iloc[0]["N_eff"], stat_df.iloc[0]["seqlen"], stat_df.iloc[0]["num_seqs"], stat_df.iloc[0]["perc_cov"]
                stats_df['DMS_ID'].append(sample)
                stats_df['bitscore'].append("0"+bitscore)
                stats_df['num_seqs'].append(num_seqs)
                stats_df['Neff/L'].append(round(neff/length, 3))
                stats_df['Colcov50'].append(perc_cov)
            except FileNotFoundError:
                continue
    return pd.DataFrame(stats_df)

In [25]:
msa_dir2_stats_df = aggregate_neff_l_colcov()

In [26]:
msa_dir2_stats_df

,DMS_ID,bitscore,num_seqs,Neff/L,Colcov50
0,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.03,10867,0.239,0.990
1,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.04,10875,0.242,0.989
2,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.05,10876,0.242,0.989
3,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.1,10877,0.241,0.989
4,A0A140D2T1_ZIKV_seqcov50_colcov50_theta0.99,0.3,10881,0.243,0.989
...,...,...,...,...,...
749,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.04,209467,1603.033,0.641
750,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.05,200782,1560.598,0.667
751,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.1,193428,1449.589,0.641
752,YNZC_BACSU_seqcov50_colcov50_theta0.8,0.3,85737,490.299,0.667


In [27]:
msa_dir2_stats_df.to_csv("/n/groups/marks/projects/marks_lab_and_oatml/ProteinGym2/EVCouplings/output/PG1_MSA_stats_Navami.csv", index=False)

### Get MSA stats from Tranception
about 90 of them are extracted by Navami. We have Spearman for 2 models and Neff/L.

## Check if, for Potts and PSSM, performance scales with Neff/L
Does it vary by coarse categories?